In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Sklearn for preprocessing and evaluation
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc, accuracy_score

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping
from scikeras.wrappers import KerasClassifier

# Set seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

In [2]:
# Load Datasets
train_df = pd.read_csv('../Data/rain_train.csv')
test_df = pd.read_csv('../Data/rain_test.csv')

# --- Feature Engineering & Preprocessing ---

def preprocess_data(df, scaler=None, is_train=True):
    df = df.copy()
    
    # Drop ID as it's not a feature
    if 'id' in df.columns:
        df = df.drop('id', axis=1)
        
    # Feature Engineering: Handle Cyclical 'day' feature
    # Using sin/cos transformation is better than raw numbers for seasonality
    df['day_sin'] = np.sin(2 * np.pi * df['day']/365.0)
    df['day_cos'] = np.cos(2 * np.pi * df['day']/365.0)
    df = df.drop('day', axis=1)
    
    X = df.drop('rainfall', axis=1)
    y = df['rainfall']

    # Scaling: Neural Networks require scaled data (mean=0, std=1) for convergence
    if is_train:
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        return X_scaled, y, scaler
    else:
        # Use the scaler fitted on training data
        X_scaled = scaler.transform(X)
        return X_scaled

# Process Train Data
X, y, scaler = preprocess_data(train_df, is_train=True)

# Split for final validation visualization (Hold-out set)
# We will use GridSearch on X_train_full, and validate visually on X_val
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training Shape: {X_train.shape}")
print(f"Validation Shape: {X_val.shape}")

Training Shape: (1752, 12)
Validation Shape: (438, 12)


In [ ]:
def create_model(learning_rate=0.001, dropout_rate=0.2, l2_reg=0.0005):
    model = Sequential()
    
    # Input Layer & First Hidden Layer
    model.add(Dense(128, input_dim=X_train.shape[1], activation='relu', 
                    kernel_regularizer=l2(l2_reg)))
    model.add(Dropout(dropout_rate))

	# Second Hidden Layer
    model.add(Dense(64, activation='relu', 
                    kernel_regularizer=l2(l2_reg)))
    model.add(Dropout(dropout_rate))

    # Third Hidden Layer
    model.add(Dense(32, activation='relu', kernel_regularizer=l2(l2_reg)))
    model.add(Dropout(dropout_rate))

    # Output Layer (Binary Classification -> Sigmoid)
    model.add(Dense(1, activation='sigmoid'))

    # Compilation
    optimizer = Adam(learning_rate=learning_rate)
    model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])
    return model

# Wrap model
model = KerasClassifier(model=create_model, verbose=0)

# Define Grid
param_grid = {
    "model__learning_rate": [0.001, 0.005, 0.01, 0.05],
    "model__dropout_rate": [0.15, 0.2, 0.25, 0.3],
    "model__l2_reg": [0.0005, 0.001, 0.005, 0.01],
    "batch_size": [16, 32, 64], 
    "epochs": [10]
}

print("Starting GridSearchCV")
grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, cv=5, scoring='accuracy', verbose=3)
grid_result = grid.fit(X_train, y_train)

print(f"Best Accuracy: {grid_result.best_score_:.4f}")
print(f"Best Parameters: {grid_result.best_params_}")

Starting GridSearchCV
Fitting 5 folds for each of 192 candidates, totalling 960 fits


In [ ]:
# Extract best params
best_params = grid_result.best_params_

# Re-build model with best params (removing the 'model__' prefix logic if needed)
# Note: KerasClassifier params have 'model__' prefix, we need to strip it for the direct function call
clean_params = {k.replace('model__', ''): v for k, v in best_params.items() if k.startswith('model__')}

final_model = create_model(**clean_params)

# Callbacks for the final training
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Train
history = final_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=best_params['batch_size'],
    callbacks=[early_stop],
    verbose=1
)

# --- PLOTTING ---
plt.figure(figsize=(12, 5))

# Plot Accuracy
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Plot Loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Predictions
y_pred_prob = final_model.predict(X_val)
y_pred = (y_pred_prob > 0.5).astype(int)

# 1. Classification Report
print("--- Classification Report ---")
print(classification_report(y_val, y_pred))

# 2. Confusion Matrix Heatmap
cm = confusion_matrix(y_val, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# 3. ROC Curve & AUC
fpr, tpr, thresholds = roc_curve(y_val, y_pred_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.show()

In [ ]:
# Process Test Data (using the scaler fitted on train data)
X_test_processed = preprocess_data(test_df, scaler=scaler, is_train=False)

# Predict
test_predictions_prob = final_model.predict(X_test_processed)
test_predictions_class = (test_predictions_prob > 0.5).astype(int).flatten()

# Update dataframe
test_df['predicted_rainfall'] = test_predictions_class

# Save to CSV
test_df.to_csv('rain_test_updated.csv', index=False)
print("Updated test.csv saved as 'rain_test_updated.csv'")

# Display first few rows
print(test_df.head())